# End-to-End ML Classification: Diabetes Health Indicators

This notebook covers data loading, EDA, preprocessing, model training, and evaluation.

## 1. Dataset Loading

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Download dataset
path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
csv_path = os.path.join(path, "diabetes_binary_health_indicators_BRFSS2015.csv")
print(f"Data Path: {csv_path}")

df = pd.read_csv(csv_path)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing value check
missing_values = df.isnull().sum()
print("Missing values in each column:\n", missing_values[missing_values > 0])

In [ ]:
# Class Balance Plot
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Diabetes_binary')
plt.title('Class Balance (Diabetes_binary)')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(18, 14))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap of All Features')
plt.show()

In [ ]:
# Distribution Plots for BMI, Age, GenHlth
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['BMI'], bins=30, ax=axes[0], kde=True)
axes[0].set_title('BMI Distribution')
sns.histplot(df['Age'], bins=13, ax=axes[1], kde=False)
axes[1].set_title('Age Distribution')
sns.countplot(data=df, x='GenHlth', ax=axes[2])
axes[2].set_title('GenHlth Distribution')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
import sys
# Make sure we can import from model/
sys.path.append(os.path.abspath('.'))
from sklearn.model_selection import train_test_split
from model.preprocessing import preprocess_data

# Train/Test Split
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['Diabetes_binary'], random_state=42)

X_train_scaled, y_train, preprocessor = preprocess_data(df_train, fit=True)
X_test_scaled, y_test, _ = preprocess_data(df_test, fit=False, preprocessor=preprocessor)

print(f"Train shape: {X_train_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")

## 4. Model Training & Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, roc_curve

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_jobs=-1),
    "Gaussian Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, n_estimators=50)
}

results = []
trained_models = {}
y_probs = {}
y_preds = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_scaled)
    y_preds[name] = y_pred
    
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = y_pred
    y_probs[name] = y_prob
        
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        "ML Model Name": name,
        "Accuracy": acc,
        "AUC": auc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "MCC": mcc
    })
    
results_df = pd.DataFrame(results).sort_values(by="MCC", ascending=False).reset_index(drop=True)
results_df

## 5. Visualizing Performance

In [ ]:
# Metrics Comparison Bar Chart
comp_melted = results_df.melt(id_vars=["ML Model Name"], var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 6))
sns.barplot(data=comp_melted, x="Metric", y="Score", hue="ML Model Name")
plt.title('Model Metrics Comparison')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# Confusion Matrix Heatmaps (2x3 Grid)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(y_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{name}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

# Leave last cell blank
axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, m_name in enumerate(["Random Forest", "Decision Tree"]):
    model = trained_models[m_name]
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    sns.barplot(x=importances[indices][:15], y=np.array(X_train_scaled.columns)[indices][:15], ax=axes[i], palette="viridis")
    axes[i].set_title(f"{m_name} - Top 15 Features")

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve Overlay
plt.figure(figsize=(8, 6))

for name, y_prob in y_probs.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = results_df[results_df["ML Model Name"] == name]["AUC"].values[0]
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc="lower right")
plt.show()

## 6. Observations & Conclusion

- **Imbalance Handling**: The dataset is heavily imbalanced (~86% non-diabetic). Models with `class_weight="balanced"` (Logistic Regression, Decision Tree, Random Forest) generally perform better in detecting the minority class (higher recall).
- **MCC as Primary Metric**: Since accuracy is misleading (a naive model predicting all zeros gets ~86%), we use the **Matthews Correlation Coefficient (MCC)** and **AUC**. 
- **Model Comparison**:
  - *Random Forest* often yields a high AUC, capturing complex relationships while controlling variance via ensemble learning.
  - *Logistic Regression* provides a solid baseline and excellent interpretability, often competing closely on AUC.
  - *Decision Tree* overfits and has lower precision/MCC compared to its ensemble counterpart.
  - *Gaussian Naive Bayes* performs surprisingly decently on AUC due to conditional independence assumptions but struggles with calibration.
  - *KNN* can be computationally expensive and sensitive to the scale and curse of dimensionality.
- **Overall Winner**: **Random Forest** (with balanced class weights) typically achieves the best balance between Recall and Precision (highest MCC) and top AUC score, making it the most reliable choice for this healthcare screening task.